In [ ]:
# Install required machine learning and interpretability libraries
%pip install scikit-learn xgboost shap

  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
  Using cached shap-0.51.0-cp313-cp313-win_amd64.whl.metadata (26 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl (8.0 MB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Using cached shap-0.51.0-cp313-cp313-win_amd64.whl (555 kB)
Using cached slicer-0.0.8-py3-none-any.whl (15 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Using cached tqdm-4.67.3-py3-none-a

In [ ]:
import os
import sys
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score

# Setup automatic reloading of src scripts
%load_ext autoreload
%autoreload 2

sys.path.append(os.path.abspath('../'))
from src.modeling import preprocess_insurance_data

# Load Data
df = pd.read_csv('../data/MachineLearningRating_v3.txt', sep='|')

# -------------------------------------------------------------------------
# TRACK A: CLAIM SEVERITY MODELS (Regression)
# -------------------------------------------------------------------------
print("--- Training Track A: Claim Severity Models ---")
X_train_r, X_test_r, y_train_r, y_test_r = preprocess_insurance_data(df, target_col='TotalClaims', task_type='regression')

reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost Regressor": XGBRegressor(n_estimators=100, random_state=42)
}

for name, model in reg_models.items():
    model.fit(X_train_r, y_train_r)
    preds = model.predict(X_test_r)
    rmse = mean_squared_error(y_test_r, preds, squared=False)
    r2 = r2_score(y_test_r, preds)
    print(f"{name:25} -> RMSE: {rmse:11.2f} | R2 Score: {r2:.4f}")

# -------------------------------------------------------------------------
# TRACK B: CLAIM PROBABILITY MODELS (Classification)
# -------------------------------------------------------------------------
print("\n--- Training Track B: Claim Probability Models ---")
X_train_c, X_test_c, y_train_c, y_test_c = preprocess_insurance_data(df, target_col='TotalClaims', task_type='classification')

cls_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost Classifier": XGBClassifier(n_estimators=100, random_state=42)
}

for name, model in cls_models.items():
    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)
    acc = accuracy_score(y_test_c, preds)
    f1 = f1_score(y_test_c, preds, zero_division=0)
    print(f"{name:25} -> Accuracy: {acc:.2%} | F1-Score: {f1:.4f}")